# Improve your LLM judge with human feedback

[Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/feedback-loop-eval.ipynb)

Should a community moderator flag “that punchline killed me”? You will score synthetic posts with a baseline moderation rubric, inspect its disagreements with example reviewer labels, and try a revised rubric on the same posts plus a held-out set. You need a Future AGI account with credentials for hosted evaluations. The notebook creates two named eval templates in your workspace.

## Install and connect

Use a fresh Python 3.12 kernel. Set the two credentials in your environment, or enter them at the hidden prompts.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "agent-learning-kit==0.3.7"], check=True);

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")

## 1. Define the baseline rubric

The baseline is deliberately literal: flag harm-related language even if it might be figurative. Give the template a unique name so you can revisit both versions without changing an existing eval.

In [ ]:
from uuid import uuid4
from fi.evals import EvalTemplateManager, Evaluator

manager, evaluator = EvalTemplateManager(), Evaluator()
run_name = f"moderation_{uuid4().hex[:8]}"
model = "turing_small"

def rubric(version, instructions):
    return manager.create_template(name=f"{run_name}_{version}", instructions=instructions,
                                   model=model, output_type="pass_fail")

v1 = rubric("v1", """Review this community post: {{post}}.
Mark FAIL if it mentions killing, death or harm, even as a figure of speech,
or if it threatens or harasses a person. Otherwise mark PASS. Explain briefly.""")
print("Baseline:", v1.name)

## 2. Score the posts before looking at the labels

Both versions see only the post text. Keep the model fixed and retain the eval ID and reason, not just the verdict.

In [ ]:
calibration = {
    "joke": "That punchline killed me. Best comedy set all year!",
    "threat": "Sam, I will hurt you if you come back here.",
    "praise": "You did a great job organizing the hike.",
}
held_out = {
    "idiom": "I'm dying of laughter at that hat on my dog.",
    "targeted": "I will hurt you for posting that, Alex.",
    "criticism": "The jacket is not worth its price.",
}

def score_posts(template, posts):
    scored = {}
    for key, post in posts.items():
        result = evaluator.evaluate(eval_templates=template.name, model_name=model,
                                    inputs={"post": post}).eval_results[0]
        verdict = str(result.output).strip().lower()
        if verdict not in {"passed", "failed", "pass", "fail"} or not result.eval_id:
            raise RuntimeError(f"No usable verdict for {key}: {result.reason}")
        scored[key] = {"safe": verdict in {"pass", "passed"},
                       "reason": result.reason, "id": str(result.eval_id)}
    return scored

baseline = score_posts(v1, calibration)

## 3. Review disagreements

These are **illustrative annotations**, not labels from a live reviewer. In your own project, have reviewers label posts independently in an [annotation queue](https://docs.futureagi.com/docs/cookbook/quickstart/dataset-annotation). Export their decisions and reasons, then match them to the model results by post ID. Keep validation posts out of the rubric-writing discussion.

In [ ]:
reviews = {
    "joke": (True, "The target is a punchline. No one is threatened."),
    "threat": (False, "This is a direct threat against a person."),
    "praise": (True, "Ordinary praise for the organizer."),
}
assert baseline.keys() == reviews.keys()
for key, (safe, note) in reviews.items():
    if safe != baseline[key]["safe"]:
        print(key, "| judge:", baseline[key]["reason"], "| reviewer:", note)

## 4. Revise the rule, not the answers

The reviewer distinguishes a figure of speech from a threat with a target. Encode that distinction as a general rule without copying the posts. Keep the original eval so you can compare decisions side by side.

In [ ]:
v2 = rubric("v2", """Review this community post: {{post}}.
Mark FAIL for a credible threat or targeted harassment. Consider intent and the target.
Mark PASS for clearly figurative praise or humor and criticism of a product rather than a person.
Do not fail a post solely for an isolated word. Explain your decision briefly.""")
revised = score_posts(v2, calibration)
print("Revised:", v2.name)

## 5. Measure fixes and regressions

Score new posts with both versions, then compare each against its reference label. A higher score on the examples used to write a rule is not evidence that it generalizes.

In [ ]:
validation_labels = {"idiom": True, "targeted": False, "criticism": True}
v1_new = score_posts(v1, held_out)
v2_new = score_posts(v2, held_out)

def compare(title, truth, before, after):
    assert truth.keys() == before.keys() == after.keys()
    print(title)
    old_hits = new_hits = 0
    for key, safe in truth.items():
        old_ok, new_ok = before[key]["safe"] == safe, after[key]["safe"] == safe
        old_hits += old_ok
        new_hits += new_ok
        change = "fixed" if new_ok and not old_ok else "regressed" if old_ok and not new_ok else "unchanged"
        print(f"  {key}: expected safe={safe}, v1={before[key]['safe']}, v2={after[key]['safe']} ({change})")
    print(f"  Agreement: {old_hits}/{len(truth)} -> {new_hits}/{len(truth)}")

compare("Calibration", {k: v[0] for k, v in reviews.items()}, baseline, revised)
compare("Held out", validation_labels, v1_new, v2_new)

## 6. Reopen the stored decisions

Look up each template and result by ID instead of running the judge again. Search for the printed template names in **Evals** to inspect your versions. Before adopting a revised rule, expand the held-out set, review any regressions, and look at false negatives as well as aggregate agreement.

In [ ]:
for template, groups in ((v1, (baseline, v1_new)), (v2, (revised, v2_new))):
    assert manager.get_template(template.id).name == template.name
    for group in groups:
        for row in group.values():
            stored = evaluator.get_eval_result(row["id"])["result"]["result"]
            assert str(stored["eval_id"]) == row["id"] and stored["name"] == template.name
            assert stored["reason"]
    print(template.name, "stored decisions:", sum(map(len, groups)))